In [ ]:
# Base imports
import os
import pickle
import re

# Compute imports
import numpy as np
import pandas as pd
import scipy
from tqdm.notebook import tqdm, trange

# Plotting imports
import matplotlib
from matplotlib import pyplot as plt
import seaborn as sns
from plotly import express as px
import matplotlib.patches as mpatches

# ML import
from sklearn.decomposition import NMF
from sklearn.metrics import mean_squared_error, median_absolute_error
from sklearn.metrics.pairwise import cosine_similarity

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['svg.fonttype'] = 'none'
matplotlib.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rcParams['font.family'] = 'sans-serif'
sns.set_style('ticks')
matplotlib.rcParams['text.color'] = '#000000'
matplotlib.rcParams['axes.labelcolor'] = '#000000'
matplotlib.rcParams['xtick.color'] = '#000000'
matplotlib.rcParams['ytick.color'] = '#000000'

In [ ]:
DF_GENES = '../../data/processed/panaroo_output/gene_presence_absence.Rtab'
ENRICHED_METADATA = '../../data/metadata/enriched_metadata.csv'
DF_EGGNOG = '../../data/processed/df_eggnog.csv'

DF_CORE_COMPLETE = '../../data/processed/CAR_genomes/df_core_panaroo.pickle'
DF_ACC_COMPLETE = '../../data/processed/CAR_genomes/df_acc_panaroo.pickle'
DF_RARE_COMPLETE = '../../data/processed/CAR_genomes/df_rare_panaroo.pickle'

L_NORM = '../../data/processed/nmf-outputs/L_norm.csv'
A_NORM = '../../data/processed/nmf-outputs/A_norm.csv'

L_BINARIZED = '../../data/processed/nmf-outputs/L_binarized.csv'
A_BINARIZED = '../../data/processed/nmf-outputs/A_binarized.csv'
L_MATRIX = '../../data/processed/nmf-outputs/L.csv'
A_MATRIX = '../../data/processed/nmf-outputs/A.csv'

In [ ]:
gene_locs_acc = pd.read_csv('acc_gene_location.csv', index_col=0)
gene_locs = pd.read_csv('complete_gene_location.csv', index_col=0)

In [ ]:
df_rare = pd.read_pickle(DF_RARE_COMPLETE)
df_acc = pd.read_pickle(DF_ACC_COMPLETE)
df_core = pd.read_pickle(DF_CORE_COMPLETE)

In [ ]:
metadata = pd.read_csv(ENRICHED_METADATA, index_col=0, dtype='object')

display( metadata.shape, metadata.head())

In [ ]:
# Load in (full) P matrix
df_genes = pd.read_csv(DF_GENES, sep='\t', index_col='Gene')

# Filter metadata for Complete sequences only
metadata_complete = metadata[metadata.genome_status == 'Complete'] # filter for only Complete sequences

# Filter P matrix for Complete sequences only
df_genes_complete = df_genes[metadata_complete.genome_id].copy()
df_genes_complete.fillna(0, inplace=True) # replace N/A with 0
df_genes_complete = df_genes_complete.astype('int8') # densify & typecast to int8 for space and compute reasons
inCompleteseqs = df_genes_complete.sum(axis=1) > 0 # filter for genes found in complete sequences
df_genes_complete = df_genes_complete[inCompleteseqs]

df_genes_complete.shape

In [ ]:
# Load in eggNOG annotations
df_eggnog = pd.read_csv(DF_EGGNOG, index_col=0)
df_eggnog.fillna('-', inplace=True)

display(
    df_eggnog.shape,
    df_eggnog.head()
)

In [ ]:
# Load in A_binarized matrix
A_binarized = pd.read_csv(A_BINARIZED, index_col=0)
A_binarized

In [ ]:
# Load in L_binarized matrix
L_binarized = pd.read_csv(L_BINARIZED, index_col=0)
L_binarized

In [ ]:
phylon_order = [
    'mobile-1',
    'mobile-4',
    'mobile-2',
    'mobile-3',
    'mobile-10',
    'mobile-7',
    'mobile-6',
    'mobile-5',
    'mobile-8',
    'mobile-9',
    'roggenkampii',
    'asburiae-1',
    'asburiae-2',
    'cancerogenous',
    'kobei',
    'bugandensis',
    'mori',
    'ludwigii',
    'cloacae',
    'hormaechei-steigerwaltii-2',
    'hormaechei-steigerwaltii-4',
    'hormaechei-steigerwaltii-1',
    'hormaechei-steigerwaltii-3',
    'hormaechei-hoffmannii-1',
    'hormaechei-hoffmannii-2',
    'hormaechei-hoffmannii-3',
    'hormaechei-hormaechei',
    'hormaechei-oharae',
    'hormaechei-xiangfangensis-2',
    'hormaechei-xiangfangensis-1',
    'hormaechei-xiangfangensis-3',
]

In [ ]:
gene_order = []

# Add in zero-phylon genes
zero_cond = L_binarized.sum(axis=1) == 0
gene_order.extend(L_binarized[zero_cond].index)

# Add in single-phylon genes
for phylon in phylon_order:
    single_cond = L_binarized.sum(axis=1) == 1
    inPhylon = L_binarized[phylon] == 1
    gene_order.extend(L_binarized[inPhylon & single_cond].index)

# Add in poly-phylon genes
for num_active_phylons in trange(2, int(L_binarized.sum(axis=1).max())+1):
    num_cond = L_binarized.sum(axis=1) == num_active_phylons
    gg = sns.clustermap(L_binarized[num_cond], method='ward', metric='euclidean', col_cluster=False, yticklabels=False);
    gene_order.extend(gg.data2d.index)

In [ ]:
# Main sorted clustermap

g = sns.clustermap(
    L_binarized.loc[gene_order],
    method='ward',
    metric='euclidean',
    row_cluster=False,
    yticklabels=False,
    cmap='Greys'
);

In [ ]:
strain_order = []
unchar_strain_order = []


# zero-phylon strains
noPhylon = A_binarized.sum() == 0
strain_order.extend(A_binarized.sum()[noPhylon].index.tolist())

# strain lists
single_phylon_strains = A_binarized.sum()[A_binarized.sum() == 1].index
multi_phylon_strains = A_binarized.sum()[A_binarized.sum() > 1].index

for phylon in phylon_order:
    if 'unchar' in phylon:
        continue
    else:
        phylon_aff_binarized_single = A_binarized.loc[phylon, single_phylon_strains]
        phylon_aff_binarized_multi = A_binarized.loc[phylon, multi_phylon_strains]
    
        inPhylon_single = phylon_aff_binarized_single == 1
        inPhylon_multi = phylon_aff_binarized_multi == 1
    
        list1 = phylon_aff_binarized_single[inPhylon_single].index.tolist()
        list2 = phylon_aff_binarized_multi[inPhylon_multi].index.tolist()
        new_list2 = list(set(list2) - set(strain_order)) # ensures no double-counting
        
        strain_order.extend(list1)
        strain_order.extend(new_list2)

for phylon in phylon_order: # must be done after the first loop
    if 'unchar' in phylon:
        phylon_aff_binarized_single = A_binarized.loc[phylon, single_phylon_strains]
        phylon_aff_binarized_multi = A_binarized.loc[phylon, multi_phylon_strains]
    
        inPhylon_single = phylon_aff_binarized_single == 1
        inPhylon_multi = phylon_aff_binarized_multi == 1
    
        list1 = phylon_aff_binarized_single[inPhylon_single].index.tolist()
        list2 = phylon_aff_binarized_multi[inPhylon_multi].index.tolist()
        new_list1 = list(set(list1) - set(strain_order)) # ensures no double-counting
        new_list2 = list(set(list2) - set(strain_order)) # ensures no double-counting
        
        strain_order.extend(new_list1)
        strain_order.extend(new_list2)

strain_order += unchar_strain_order

# A-binarized
sns.clustermap(A_binarized.loc[phylon_order, strain_order], cmap='Greys', xticklabels=False, row_cluster=False, col_cluster=False)

In [ ]:
characterized_order = [x for x in phylon_order if 'mobile' not in x]

In [ ]:
custom_colors = [
    # non hormaechei species
    "Green",
    'Blue',
    'navy',
    'Magenta',
    'Purple',
    'Cyan',
    'Tan',
    'Lime',
    'Pink',
    # hormaechei colors
    'firebrick',
    'maroon',
    'darkred',
    'brown',
    'Goldenrod',
    'DarkGoldenrod',
    'Gold',
    'Yellow',
    'Red',
    'Orange',
    'darkorange',
    'orangered',
]

In [ ]:
def get_strains(phylon, A_binarized = A_binarized):
    phylon_membership = A_binarized.loc[phylon]
    return (phylon_membership[phylon_membership == 1]).index

# Analyze AMR Distribution across phylons

In [ ]:
amr = pd.read_csv('../../data/processed/amrfinder/output', sep = '\t')
amr['Protein identifier'] = amr['Protein identifier'].apply(lambda x: x)
amr = amr.sort_values('% Coverage of reference sequence')
amr = amr.drop_duplicates(subset='Protein identifier', keep="last")

In [ ]:
acc_amr = [x for x in amr['Protein identifier'] if x in df_acc.index]
core_amr = [x for x in amr['Protein identifier'] if x in df_core.index]
rare_amr = [x for x in amr['Protein identifier'] if x in df_rare.index]

In [ ]:
df_amr_percentages = pd.DataFrame(index = core_amr + acc_amr + rare_amr, columns=characterized_order)

for phylon in characterized_order:
    phylon_strains = get_strains(phylon)
    df_amr_percentages[phylon] = df_genes_complete.loc[core_amr + acc_amr + rare_amr, phylon_strains].sum(axis=1) / len(phylon_strains)

sns.clustermap(df_amr_percentages[df_amr_percentages.sum(axis=1) > 0].loc[:,characterized_order], col_cluster=False, row_cluster=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get unique subclasses
unique_subclasses = amr['Class'].unique()

# If you want at least 20 distinct colors, you can sample from a colormap
num_colors = max(20, len(unique_subclasses))  # Ensure at least 20 colors

# Generate a colormap
colors = plt.cm.get_cmap("tab20", num_colors)  # 'tab20' is a colormap with 20 distinct colors

# Sample the colors
color_list = [colors(i) for i in range(num_colors)]

# If there are more than 20 subclasses, you can repeat or modify the colors
if len(unique_subclasses) > num_colors:
    repeat_count = (len(unique_subclasses) // num_colors) + 1
    color_list = color_list * repeat_count  # Repeat the colors as necessary
    color_list = color_list[:len(unique_subclasses)]  # Trim to the exact number of unique subclasses

# Zip the unique subclasses with the colors
subclass_color_map = dict(zip(unique_subclasses, color_list))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Initialize the figure with your desired size
plt.figure(figsize=(8.5, 20))

# Define a scaling factor to space the points further apart
scaling_factor = 150  # Adjust this value to increase or decrease the spacing

# Create a list of x-positions for each gene, spaced further apart
x_positions = []
x_positions = x_positions + [i*scaling_factor for i in range(len(core_amr))]
x_positions = x_positions + [max(x_positions) + 1000 + i*scaling_factor for i in range(len(acc_amr))]
x_positions = x_positions + [max(x_positions) + 1000 + i*scaling_factor for i in range(len(rare_amr))]
x_positions = [abs(x - max(x_positions)) for x in x_positions]
x_positions = x_positions[::-1]

plt.axhline(x_positions[-len(core_amr)] - 500, c = 'k', linestyle='dashed')
plt.text(-.5, x_positions[-len(core_amr)] + 500, "Core AMR Genes")

plt.axhline(x_positions[-(len(core_amr) + len(acc_amr))] - 500, c = 'k', linestyle='dashed')
plt.text(-.5, x_positions[-len(core_amr)] - 700, "Acc AMR Genes")

plt.text(-.5, x_positions[-(len(core_amr) + len(acc_amr))] - 700, "Rare AMR Genes")

# Loop over the genes and plot them
for i, gene in enumerate(rare_amr + acc_amr + core_amr):
    subclass = amr.set_index('Protein identifier').loc[gene, 'Class']
    
    # Assign the x position based on the index of the gene
    x_pos = x_positions[i]

    edgecolors = []
    for phylon in characterized_order:
        if (gene in L_binarized.index):
            if  (L_binarized.loc[gene, phylon] > 0):
                edgecolors.append('k')
            else:
                edgecolors.append(subclass_color_map[subclass])
        else:
            edgecolors.append(subclass_color_map[subclass])
    
    # Plot the scatter points with the new x-positions
    plt.scatter(df_amr_percentages.columns,
                [x_pos] * len(df_amr_percentages.columns),  
                s=df_amr_percentages.loc[gene]**2 * 100, 
                alpha=0.6, 
                label=gene, 
                c=subclass_color_map[subclass],
                edgecolors=edgecolors
               )




# Customize the plot
plt.title("Bubble Plot of AMR Gene Presence Across Phylons")


plt.xlabel("Phylon")
plt.xticks(rotation = 90)

plt.ylabel("Gene Index")
plt.yticks([])

legend_elements = []

legend_elements.append(Line2D([0], [0], marker='o', color='w', label="Antibiotic Class",
                          markerfacecolor='w', markersize=0, linewidth=0))
for key, val in subclass_color_map.items():
    legend_elements.append(Line2D([0], [0], marker='o', color=val, label=key,
                          markerfacecolor=val, markersize=12, linewidth=0))

legend_elements.append(Line2D([0], [0], marker='o', color='w', label="Percentage of Strains with Gene",
                          markerfacecolor='w', markersize=0, linewidth=0))

plt.legend(handles = legend_elements, loc='upper right', bbox_to_anchor=(1.6, 1))



# Show the plot
plt.show()


# Separate Plots for Core/Acc and Rare Genomes

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get unique subclasses
unique_subclasses = amr['Class'].unique()

# If you want at least 20 distinct colors, you can sample from a colormap
num_colors = max(20, len(unique_subclasses))  # Ensure at least 20 colors

# Generate a colormap
colors = plt.cm.get_cmap("tab20", num_colors)  # 'tab20' is a colormap with 20 distinct colors

# Sample the colors
color_list = [colors(i) for i in range(num_colors)]

# If there are more than 20 subclasses, you can repeat or modify the colors
if len(unique_subclasses) > num_colors:
    repeat_count = (len(unique_subclasses) // num_colors) + 1
    color_list = color_list * repeat_count  # Repeat the colors as necessary
    color_list = color_list[:len(unique_subclasses)]  # Trim to the exact number of unique subclasses

# Zip the unique subclasses with the colors
subclass_color_map = dict(zip(unique_subclasses, color_list))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Initialize the figure with your desired size
plt.figure(figsize=(5, 12))

# Define a scaling factor to space the points further apart
scaling_factor = 200  # Adjust this value to increase or decrease the spacing

# Create a list of x-positions for each gene, spaced further apart
x_positions = []
x_positions = x_positions + [i*scaling_factor for i in range(len(core_amr))]
x_positions = x_positions + [max(x_positions) + 400 + i*scaling_factor for i in range(len(acc_amr))]
x_positions = [abs(x - max(x_positions)) for x in x_positions]
x_positions = x_positions[::-1]

plt.axhline(x_positions[-len(core_amr)] - 200, c = 'k', linestyle='dashed')
plt.text(-.5, x_positions[-len(core_amr)] + 270, "Core AMR Genes")

plt.text(-.5, x_positions[-len(core_amr)] - 310, "Acc AMR Genes")

sizes = []
# Loop over the genes and plot them
for i, gene in enumerate(acc_amr + core_amr):
    subclass = amr.set_index('Protein identifier').loc[gene, 'Class']
    
    # Assign the x position based on the index of the gene
    x_pos = x_positions[i]

    edgecolors = []
    for phylon in characterized_order:
        if (gene in L_binarized.index):
            if  (L_binarized.loc[gene, phylon] > 0):
                edgecolors.append('k')
            else:
                edgecolors.append('w')
        else:
            edgecolors.append('w')

    # Plot the scatter points with the new x-positions
    plt.scatter(df_amr_percentages.columns,
                [x_pos] * len(df_amr_percentages.columns),  
                s=df_amr_percentages.loc[gene]**2 * 100, 
                alpha=0.9, 
                label=gene, 
                c=subclass_color_map[subclass],
                edgecolors=edgecolors
               )
    sizes += (df_amr_percentages.loc[gene]**2 * 100).to_list()



# Customize the plot
plt.title("Bubble Plot of AMR Gene Presence Across Phylons")


plt.xlabel("Phylon")
plt.xticks(rotation = 90)

plt.ylabel("Gene Index")
plt.yticks([])

legend_elements = []

legend_elements.append(Line2D([0], [0], marker='o', color='w', label="Antibiotic Class",
                          markerfacecolor='w', markersize=0, linewidth=0))

for key, val in subclass_color_map.items():
    legend_elements.append(Line2D([0], [0], marker='o', color=val, label=key,
                          markerfacecolor=val, markersize=12, linewidth=0))


legend_elements.append(Line2D([0], [0], marker='o', color='w', label="",
                          markerfacecolor='w', markersize=0, linewidth=0))
legend_elements.append(Line2D([0], [0], marker='o', color='w', label="Percentage of Strains with Gene",
                          markerfacecolor='w', markersize=0, linewidth=0))

sizes = sorted(sizes)

# Calculate percentiles of sizes
percentiles = [max(sizes), max(sizes) * .75, max(sizes) * .5, max(sizes) * .25]

# Create the legend entries for the size distribution
size_legend_elements = []
for percentile in percentiles:
    legend_size = (percentile ** 2) * 100
    size_legend_elements.append(Line2D([0], [0], marker='o', color='w', 
                                        label=f'{int(percentile)}% Percent Presence',
                                        markerfacecolor='gray', markersize=np.sqrt(legend_size) * 0.011))  # Adjust the scaling as needed

# Append the size legend elements to the main legend
legend_elements += size_legend_elements

# Customize the plot again (this step may be redundant but keeps the structure clear)
plt.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(2, 1))

# Show the plot
plt.show()


In [ ]:
df_genes_complete.loc[gene] > 0

In [ ]:
rare_classes = amr[amr['Protein identifier'].isin(df_rare.index)]['Class'].unique()

rare_plot_df = pd.DataFrame(index = rare_classes, columns = df_genes_complete.columns, dtype=int).fillna(0)
for gene in rare_amr:
    subclass = amr.set_index('Protein identifier').loc[gene, 'Class']
    rare_plot_df.loc[subclass] += df_genes_complete.loc[gene] > 0

rare_plot_df = rare_plot_df.astype(bool).astype(int) # set max for each strain in each class to 1

rare_phylons_amr_df = pd.DataFrame(index = rare_classes, columns = characterized_order, dtype=int).fillna(0)

for phylon in characterized_order:
    p_strains = get_strains(phylon)
    for subclass in rare_plot_df.index:
        rare_phylons_amr_df.loc[subclass, phylon] = rare_plot_df.loc[subclass, p_strains].sum() / len(p_strains)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Initialize the figure with your desired size
plt.figure(figsize=(5, 12))

# Define a scaling factor to space the points further apart
scaling_factor = 200  # Adjust this value to increase or decrease the spacing

# Create a list of x-positions for each gene, spaced further apart
x_positions = []
x_positions = x_positions + [i*scaling_factor for i in range(len(rare_phylons_amr_df))]

# Loop over the genes and plot them
for i, subclass in enumerate(rare_phylons_amr_df.index):
    # Assign the x position based on the index of the gene
    x_pos = x_positions[i]

    edgecolors = []
    for phylon in characterized_order:
        edgecolors.append(subclass_color_map[subclass])
    
    # Plot the scatter points with the new x-positions
    plt.scatter(rare_phylons_amr_df.columns,
                [x_pos] * len(df_amr_percentages.columns),  
                s=rare_phylons_amr_df.loc[subclass]**2 * 100, 
                alpha=0.6, 
                label=gene, 
                c=subclass_color_map[subclass],
                edgecolors=edgecolors
               )

# Customize the plot
plt.title("Rare AMR Gene Presence Across Phylons")


plt.xlabel("Phylon")
plt.xticks(rotation = 90)

plt.ylabel("Gene Index")
plt.yticks([])

legend_elements = []

legend_elements.append(Line2D([0], [0], marker='o', color='w', label="Antibiotic Class",
                          markerfacecolor='w', markersize=0, linewidth=0))

for key, val in subclass_color_map.items():
    legend_elements.append(Line2D([0], [0], marker='o', color=val, label=key,
                          markerfacecolor=val, markersize=12, linewidth=0))


legend_elements.append(Line2D([0], [0], marker='o', color='w', label="",
                          markerfacecolor='w', markersize=0, linewidth=0))
legend_elements.append(Line2D([0], [0], marker='o', color='w', label="Percentage of Strains with Gene",
                          markerfacecolor='w', markersize=0, linewidth=0))

sizes = sorted(sizes)

# Calculate percentiles of sizes
percentiles = [max(sizes), max(sizes) * .75, max(sizes) * .5, max(sizes) * .25]

# Create the legend entries for the size distribution
size_legend_elements = []
for percentile in percentiles:
    legend_size = (percentile ** 2) * 100
    size_legend_elements.append(Line2D([0], [0], marker='o', color='w', 
                                        label=f'{int(percentile)}% Percent Presence',
                                        markerfacecolor='gray', markersize=np.sqrt(legend_size) * 0.011))  # Adjust the scaling as needed

# Append the size legend elements to the main legend
legend_elements += size_legend_elements

# Customize the plot again (this step may be redundant but keeps the structure clear)
plt.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(2, 1))

# Show the plot
plt.show()

# Accessory Plot with L_norm

In [ ]:
L_norm = pd.read_csv(L_NORM, index_col=0)
L_norm.loc['ampC~~~blaACT~~~blaACT37~~~blaACT154~~~blaACT64~~~blaACT143~~~blaACT102~~~blaACT52~~~blaCMH~~~blaCMH24~~~blaMIR~~~blaMIR5~~~blaACT2~~~blaACT170~~~blaACT145~~~blaACT84~~~blaACT17'] = [1] * len(L_norm.columns)
L_norm = L_norm.loc[:,characterized_order]
L_norm = L_norm.loc[core_amr + acc_amr][L_norm.loc[core_amr + acc_amr].max(axis=1) > .5]

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.figure(figsize=(5.5,6))  # Set figure size to 7x6 inches


scaling_factor = 50

# Define the custom order for sorting subclasses
custom_order = ['TRIMETHOPRIM', 'TETRACYCLINE', 'SULFONAMIDE', 'QUINOLONE', 'QUATERNARY AMMONIUM', 'MACROLIDE', 
                'BETA-LACTAM', 'AMINOGLYCOSIDE', 'FOSFOMYCIN', 'PHENICOL/QUINOLONE', 'PHENICOL']

# Create a dictionary that maps subclass to its custom order (numeric value)
subclass_order = {subclass: idx for idx, subclass in enumerate(custom_order)}

# Get the accessory genes and their subclasses
acc_genes = list(L_norm.iloc[1:].index)  # Accessory genes
subclasses = amr.set_index('Protein identifier').loc[acc_genes + core_amr, 'Class']

# Create a list of tuples (gene, subclass) and sort the accessory genes based on the custom order
sorted_acc_genes = sorted(acc_genes, key=lambda gene: subclass_order.get(subclasses[gene], len(custom_order)))

# Combine core genes (without sorting) and sorted accessory genes
sorted_genes = core_amr + sorted_acc_genes

# Recompute x_positions for sorted genes
x_positions = []
x_positions += [i * scaling_factor for i in range(len(core_amr))]  # For core genes
x_positions += [max(x_positions) + 100 + i * scaling_factor for i in range(len(sorted_acc_genes))]  # For sorted accessory genes
x_positions = [abs(x - max(x_positions)) for x in x_positions]
# Reverse x_positions for better placement (core genes at top, accessory genes below)
# x_positions = x_positions[::-1]

# Create ytick labels for sorted genes, starting with the core genes followed by the accessory genes
ytick_labels = [subclasses[gene] for gene in sorted_genes]
ytick_positions = x_positions

# Apply yticks
plt.yticks(ytick_positions, ytick_labels, fontsize=10)

# Add labels and lines for core and accessory genes
gap_position = x_positions[len(core_amr)] + 50  # Position for the gap (horizontal line) at the top of the accessory genes

# Horizontal dashed line indicating the gap between core and accessory genes
plt.axhline(gap_position, c='k', linestyle='dashed')

# Labels for core and accessory genes
plt.text(-.5, gap_position + 20, "Core AMR Genes")
plt.text(-.5, gap_position - 35, "Acc AMR Genes")

sizes = []

# Loop over the genes and plot them
for i, gene in enumerate(sorted_genes):
    subclass = subclasses[gene]  # Get the subclass of the gene
    
    # Assign the x position based on the index of the gene
    x_pos = x_positions[i]

    edgecolors = []
    for phylon in characterized_order:
        if gene in L_binarized.index:
            if L_binarized.loc[gene, phylon] > 0:
                edgecolors.append('k')
            else:
                edgecolors.append('w')
        else:
            edgecolors.append('k')

    if L_norm.loc[gene].max() > 0.4:
        # Plot the scatter points with the new x-positions
        plt.scatter(L_norm.columns,
                    [x_pos] * len(L_norm.columns),  
                    s=L_norm.loc[gene]**2 * 100, 
                    alpha=0.9, 
                    label=gene, 
                    c=custom_colors,
                    edgecolors=edgecolors,
                    linewidths=.6
                   )
        sizes += (L_norm.loc[gene]**2 * 100).to_list()

# Customize the plot
plt.title("Bubble Plot of AMR Gene Presence Across Phylons")
plt.xlabel("Phylon")
plt.xticks([])  # Remove x-ticks
plt.ylabel("Gene Index")

# Set yticks and labels for the sorted order
plt.yticks(ytick_positions, ytick_labels, fontsize=10)

# Show the plot
plt.savefig('../images/fig3/AMR_Dot_new.svg', format='svg', dpi=300, bbox_inches='tight')
plt.show()